# [실습] LangChain을 이용한 데이터 생성과 처리


LCEL의 기본 문법인 Prompt | llm | Parser 구조에 대해 배웠습니다.   
이번 실습에서는 출력을 구조화하고, LLM을 연결하는 방법에 대해 알아봅니다.


### 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [1]:
%pip install langchain langchain_openai dotenv rich -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

OpenAI API 키 확인


### LLM 모델 불러오기

In [3]:
from langchain.chat_models import init_chat_model

gpt_llm = init_chat_model(
    "gpt-5.2", reasoning_effort='low')

## JsonOutputParser 로 Json 형식의 출력 만들기

LLM의 출력을 구조화하면, 데이터 후처리를 하지 않고도 다른 코드와 연결할 수 있습니다.   
JSON 형식의 출력을 구성해 보겠습니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

jsonparser = JsonOutputParser()

JSON 파서의 역할은 JSON 규격에 맞는 텍스트를 Dict 형식으로 변환하는 것으로,     
실제 형식에 대한 조건을 프롬프트로 전달해야 합니다.

In [5]:
jsonparser.get_format_instructions()

'Return a JSON object.'

In [6]:
recipe_template = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료와 조건을 이용한 환상적인
퓨전 다이닝을 만들고 싶습니다. 1가지 메뉴만 추천해주세요!
레시피에 대한 정보를 JSON 형식으로 출력해주세요.

[재료]: {ingredient}
[조건]: {condition}
''')
])

recipe_chain = recipe_template | gpt_llm | jsonparser

In [7]:
response = recipe_chain.invoke({'ingredient':'콜라, 고수, 새우', 'condition':'디저트'})
response

{'menu_name': '콜라 카라멜 쉬림프 프랄린 & 고수-라임 그라니타',
 'type': 'dessert',
 'fusion_concept': '콜라를 졸여 만든 카라멜(탄산 향+스파이스 뉘앙스)에 새우를 바삭하게 코팅해 프랄린처럼 만들고, 고수의 허브향을 라임과 함께 얼려 그라니타로 곁들이는 달콤-짭짤-상큼 퓨전 디저트',
 'servings': 2,
 'ingredients': [{'group': '콜라 카라멜 쉬림프',
   'items': [{'name': '새우(껍질 제거, 내장 제거)', 'amount': '10마리(중하 크기)'},
    {'name': '콜라', 'amount': '250ml'},
    {'name': '버터', 'amount': '20g'},
    {'name': '갈색설탕(선택)', 'amount': '1큰술'},
    {'name': '소금', 'amount': '꼬집'},
    {'name': '라임 제스트(선택)', 'amount': '약간'},
    {'name': '전분(감자/옥수수)', 'amount': '2큰술'},
    {'name': '식용유', 'amount': '적당량(팬에 0.5cm)'}]},
  {'group': '고수-라임 그라니타',
   'items': [{'name': '물', 'amount': '200ml'},
    {'name': '설탕', 'amount': '35g'},
    {'name': '라임즙', 'amount': '2큰술'},
    {'name': '고수(잎 위주)', 'amount': '한 줌(약 15g)'},
    {'name': '소금', 'amount': '아주 약간'}]},
  {'group': '마무리(선택)',
   'items': [{'name': '화이트초콜릿(얇게 깎기)', 'amount': '10g'},
    {'name': '볶은 코코넛칩 또는 깨', 'amount': '1큰술'}]}],
 'steps': [{'part': '고수-라

In [8]:
# Dict 구조: 추출 가능
response['menu_name']

'콜라 카라멜 쉬림프 프랄린 & 고수-라임 그라니타'

Json으로 파싱하는 방법은 활용도가 높지만, 실행할 때마다 결과뿐만 아니라 형식도 달라진다는 문제가 있습니다.

In [9]:
response = recipe_chain.invoke({'ingredient':'문어, 피넛버터', 'condition':'메인 요리'})
response

{'menu_name_ko': '피넛버터 라케로 글레이즈드 문어 스테이크',
 'menu_name_en': 'Peanut-Butter Lacquered Octopus Steak',
 'category': '메인 요리',
 'fusion_concept': {'cuisines': ['한국', '서아프리카(마페 감성)', '일본(테리야키식 글레이즈)'],
  'idea': "피넛버터를 고소한 소스 베이스로 쓰고, 간장·식초·고추로 밸런스를 맞춘 뒤 문어에 '라케(코팅) 글레이즈'를 여러 번 발라 윤기와 풍미를 극대화한 메인 디시"},
 'servings': 2,
 'difficulty': '중',
 'time': {'prep_minutes': 15, 'cook_minutes': 35, 'total_minutes': 50},
 'ingredients': [{'name': '문어(자숙 또는 생문어)',
   'amount': '450g',
   'notes': '생문어면 부드럽게 삶는 과정 포함'},
  {'name': '피넛버터(크리미)',
   'amount': '3 큰술(약 45g)',
   'notes': '무가당/가당 모두 가능(가당이면 설탕량 조절)'},
  {'name': '간장', 'amount': '2 큰술'},
  {'name': '쌀식초(또는 라임/레몬즙)', 'amount': '1.5 큰술'},
  {'name': '꿀 또는 흑설탕', 'amount': '1 큰술'},
  {'name': '고추장', 'amount': '1 큰술', 'notes': '없으면 스리라차/칠리페이스트로 대체'},
  {'name': '다진 마늘', 'amount': '1 작은술'},
  {'name': '생강 간 것', 'amount': '1 작은술'},
  {'name': '물 또는 육수', 'amount': '4~6 큰술', 'notes': '소스 농도 조절용'},
  {'name': '참기름', 'amount': '1 작은술'},
  {'name': '식용유', '

## Pydantic을 이용해 출력 형식 지정하기

pydantic은 데이터 형식에 제약조건을 두고 이를 준수하는지 검증하는 라이브러리입니다.


In [10]:
from pydantic import BaseModel, Field
# pydantic 연동

class Recipe(BaseModel):
    name: str = Field(description="음식 이름")
    # name: 문자열, 설명은 "음식 이름"
    difficulty: str = Field(description="만들기의 난이도")

    origin: str = Field(description="원산지")
    ingredients: list[str] = Field(description="재료")
    # ingredients: 문자열 리스트, 설명은 "재료"

    instructions: list[str] = Field(description="조리법")
    tip: str = Field(description='실패하는 5가지 시나리오')


In [11]:
parser = JsonOutputParser(pydantic_object=Recipe)

In [12]:
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


해당 내용을 프롬프트에 포함합니다.

In [13]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
재료: {ingredient}
출력 형식 조건: {instruction}''')
])

recipe_chain2 = recipe_template2 | gpt_llm | parser


In [14]:
recipe_chain2.invoke({'ingredient':'생강', 'instruction':parser.get_format_instructions()})

{'name': '생강 코지 버터 미소 카라멜 팝콘',
 'difficulty': '중',
 'origin': '일본 코지·미소 발효 풍미 + 서구식 카라멜 팝콘 퓨전',
 'ingredients': ['팝콘용 옥수수 100g',
  '식용유 2큰술',
  '무염버터 60g',
  '설탕 120g',
  '물엿(또는 꿀) 30g',
  '미소(된장) 1큰술',
  '생강(생강즙 또는 강판) 1~2큰술',
  '간장 1작은술',
  '베이킹소다 1/4작은술',
  '코지(건조 쌀코지) 가루 1~2큰술(선택)',
  '소금 한 꼬집',
  '검은깨 또는 김가루 약간(선택)'],
 'instructions': ['큰 냄비에 식용유를 두르고 옥수수를 넣은 뒤 뚜껑을 덮어 팝콘을 튀긴다. 다 튀겨지면 넓은 볼이나 오븐팬으로 옮긴다.',
  '두꺼운 바닥 냄비에 버터를 녹인 뒤 설탕과 물엿을 넣고 중불에서 저어가며 끓인다.',
  '시럽이 끓어오르면 불을 중약불로 낮추고 미소, 간장, 생강(즙/강판)을 넣어 완전히 풀어준다. (코지 가루를 쓸 경우 이때 넣어 잘 섞는다.)',
  '거품이 잦아들고 색이 연한 호박색이 되면 불을 끄고 베이킹소다와 소금 한 꼬집을 넣어 빠르게 섞는다. (순간적으로 부풀며 기포가 생긴다.)',
  '즉시 팝콘 위에 소스를 고루 부어 주걱으로 빠르게 버무린다. 덩어리가 생기면 손으로 살짝 부숴가며 섞는다.',
  '오븐 120~130°C에서 20~25분 정도 말리듯이 구워 바삭하게 만든다. 중간에 한 번 뒤집어 골고루 건조한다. (에어프라이어는 110~120°C로 10~15분, 중간에 흔들기.)',
  '완전히 식힌 뒤 검은깨나 김가루를 소량 뿌려 마무리한다. 밀폐 용기에 보관한다.'],
 'tip': '실패하는 5가지 시나리오: 1) 카라멜이 타서 쓴맛이 남음—불이 세거나 끓이는 시간이 길면 생강·미소 향이 날아가고 탄맛이 우세해진다. 2) 생강을 너무 많이 넣어 매운 비누맛/섬유감이 도드라짐—강판 생강은 1큰술부터 시작하고 즙을 쓰면 섬유감

partial을 통해 먼저 일부를 입력할 수도 있습니다.

In [15]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
]).partial(instruction=parser.get_format_instructions())

recipe_chain2 = recipe_template2 | gpt_llm | parser

recipe_chain2.invoke('감자')
# partial은 기본값: instruction을 다시 덮어쓸 수도 있음

{'name': '감자 코지(麹) 카라멜 미소 브륄레 + 김오일',
 'difficulty': '중상',
 'origin': '퓨전(일본 발효 × 프랑스 디저트 × 한국 김)',
 'ingredients': ['감자 500g(전분 많은 품종 추천)',
  '우유 250ml',
  '생크림 200ml',
  '달걀노른자 4개',
  '설탕 70g(커스터드용)',
  '백미소 25g',
  '소금 한 꼬집',
  '바닐라 익스트랙 1작은술(선택)',
  '코지(쌀누룩) 60g 또는 코지 파우더 25g',
  '물 60ml(코지 시럽용)',
  '설탕 80g(코지 카라멜용)',
  '버터 20g(카라멜 마무리)',
  '김 4장(또는 김가루 2큰술)',
  '중성 오일 60ml(포도씨유 등)',
  '토치용 설탕 2~3큰술(브륄레 표면)'],
 'instructions': ['감자 준비: 감자를 껍질째 찐 뒤(또는 삶은 뒤) 껍질을 벗기고 뜨거울 때 으깬다. 더 매끈하게 하려면 체에 한 번 내린다.',
  '코지 시럽(실험 포인트): 코지와 물을 소스팬에 넣고 55~60℃를 유지하며 45~60분 보온한다(불은 가장 약하게, 온도계 권장). 전분/당화가 진행되면 달큰한 향이 난다. 면포나 고운 체로 걸러 시럽만 받는다.',
  '코지 카라멜: 다른 냄비에 설탕 80g을 중불로 녹여 호박색이 되면 불을 끄고 버터를 넣어 녹인다. 여기에 코지 시럽을 2~3회 나눠 넣고(튀어 오를 수 있음) 다시 약불에서 부드럽게 풀어 카라멜 소스를 만든다. 식혀둔다.',
  '김오일: 마른 팬에 김을 10~15초만 가볍게 구워 향을 깨운 뒤 잘게 부순다. 중성 오일과 함께 60℃ 정도로 15분 데운 후(끓이지 않기) 식혀서 체로 걸러 김오일을 만든다.',
  '커스터드 베이스: 우유+생크림을 냄비에 넣고 가장자리가 살짝 끓기 직전(80℃ 전후)까지 데운다. 별볼에 노른자와 설탕 70g, 소금, 백미소를 넣고 고루 푼다.',
  '템퍼링: 데운 유제품을 노른자 볼에 조금씩 부어가며 섞은 

# LangChain Structured Output
파서를 사용하지 않고, 구조화된 출력을 생성합니다.  

In [16]:
from rich import print as rprint
structured_llm = gpt_llm.with_structured_output(Recipe)

rprint(structured_llm)

RunnableSequence(
    first=_ChatModelBinding(
        bound=ChatOpenAI(
            metadata={
                'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}
            },
            output_version=None,
            profile={
                'name': 'GPT-5.2',
                'release_date': '2025-12-11',
                'last_updated': '2025-12-11',
                'open_weights': False,
                'max_input_tokens': 272000,
                'max_output_tokens': 128000,
                'text_inputs': True,
                'image_inputs': True,
                'audio_inputs': False,
                'video_inputs': False,
                'text_outputs': True,
                'image_outputs': False,
                'audio_outputs': False,
                'video_outputs': False,
                'reasoning_output': True,
                'tool_calling': True,
                'structured_output': True,
                'attachment': True,
                'temperature': False,
                'image_url_inputs': True,
                'pdf_inputs': True,
                'pdf_tool_message': True,
                'image_tool_message': True,
                'tool_choice': True,
                'tool_call_streaming': True,
                'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']
            },
            client=<openai.resources.chat.completions.completions.Completions object at 0x000001B82E32E850>,
            async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 
0x000001B82ED0F2D0>,
            root_client=<openai.OpenAI object at 0x000001B82E32E1D0>,
            root_async_client=<openai.AsyncOpenAI object at 0x000001B82ED0EE50>,
            model_name='gpt-5.2',
            model_kwargs={},
            openai_api_key=SecretStr('**********'),
            openai_proxy=None,
            stream_usage=True,
            reasoning_effort='low',
            stream_chunk_timeout=120.0
        ),
        kwargs={
            'response_format': <class '__main__.Recipe'>,
            'ls_structured_output_format': {
                'kwargs': {'method': 'json_schema', 'strict': None},
                'schema': {
                    'type': 'function',
                    'function': {
                        'name': 'Recipe',
                        'description': '',
                        'parameters': {
                            'properties': {
                                'name': {'description': '음식 이름', 'type': 'string'},
                                'difficulty': {'description': '만들기의 난이도', 'type': 'string'},
                                'origin': {'description': '원산지', 'type': 'string'},
                                'ingredients': {
                                    'description': '재료',
                                    'items': {'type': 'string'},
                                    'type': 'array'
                                },
                                'instructions': {
                                    'description': '조리법',
                                    'items': {'type': 'string'},
                                    'type': 'array'
                                },
                                'tip': {'description': '실패하는 5가지 시나리오', 'type': 'string'}
                            },
                            'required': ['name', 'difficulty', 'origin', 'ingredients', 'instructions', 'tip'],
                            'type': 'object'
                        }
                    }
                }
            }
        },
        config={},
        config_factories=[]
    ),
    middle=[],
    last=RunnableBinding(
        bound=RunnableLambda(...),
        kwargs={},
        config={},
        config_factories=[],
        custom_output_type=<class '__main__.Recipe'>
    )
)

In [17]:
recipe_template3 = ChatPromptTemplate([
    ('system','당신은 한국 전통의 재료가 가진 다양한 맛과 특성을 활용합니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!''')
])

recipe_chain3 = recipe_template3 | structured_llm
response = recipe_chain3.invoke("생강")
rprint(response)

Recipe(
    name='생강 발효장(간장·고추장·식초) 글레이즈 두부 스테이크 & 유자생강 백김치 살사',
    difficulty='중',
    origin='한국(발효장·유자·백김치 풍미를 응용한 퓨전)',
    ingredients=[
        '단단한 두부 1모(약 300~400g)',
        '생강 40~60g(절반은 강판, 절반은 얇게 채썰기)',
        '간장 2큰술',
        '고추장 1큰술',
        '조청(또는 꿀/올리고당) 1.5큰술',
        '현미식초(또는 매실식초) 1큰술',
        '맛술 1큰술(선택)',
        '참기름 1작은술',
        '마늘 1쪽(다지기)',
        '후추 약간',
        '들기름 1작은술(마무리)',
        '깻잎 5~7장(가늘게 채썰기)',
        '유자청 1큰술',
        '백김치(또는 무피클) 1컵 분량(잘게 다지기)',
        '대파 흰 부분 약간(송송)',
        '통깨 또는 볶은 들깨가루(선택)',
        '식용유 약간',
        '소금 약간'
    ],
    instructions=[
        '두부는 키친타월로 감싸 15분 이상 눌러 물기를 빼고, 2~3cm 두께로 썬 뒤 소금을 아주 살짝 뿌려 둡니다.',
        '글레이즈 소스를 만듭니다: 간장, 고추장, 조청, 식초, (맛술), 참기름, 다진 마늘, 강판 간 생강 절반을 
섞습니다. 매운맛/산미/단맛은 1:1:1 느낌으로 취향 조절합니다.',
        '팬을 중약불로 달구고 식용유를 두른 뒤 두부를 노릇하게 굽습니다(한 면 3~4분). 뒤집어 반대면도 굽습니다.',
        '불을 약하게 줄이고 글레이즈 소스를 팬에 부어 두부에 끼얹으며 1~2분 졸여 코팅합니다. 이때 남은 채썬 생강을 
넣어 살짝만 익혀 향을 세웁니다(과익히면 쓴맛).',
        '유자생강 백김치 살사를 만듭니다: 잘게 다진 백김치(또는 무피클)에 유자청, 대파, 깻잎을 섞고, 필요하면 
식초/소금으로 간을 맞춥니다. 여기에 생강 약간을 아주 잘게 다져 넣어 ‘서늘한 매운 향’을 추가합니다.',
        '접시에 두부 스테이크를 올리고 팬의 글레이즈를 끼얹습니다. 살사를 곁들인 뒤 들기름을 한두 방울 떨어뜨리고 
통깨(또는 들깨가루)로 마무리합니다.',
        '실험 포인트를 더하고 싶다면: 글레이즈를 거의 다 졸인 뒤 불을 끄고 식초를 1작은술 추가해 향을 ‘후반에’ 
터뜨리거나, 생강을 아주 얇게 썰어 바삭하게 튀겨 토핑으로 올립니다.'
    ],
    tip='실패 시나리오 5가지: (1) 생강을 오래 볶아 쓴맛/떫은맛이 올라옴 → 생강은 ‘마지막 30~60초’만 가열. (2) 
글레이즈가 타서 쓴맛 → 조청/꿀이 들어가면 약불 유지, 팬이 너무 달궈졌으면 불 끄고余열로 코팅. (3) 두부가 물러져 
코팅이 안 됨 → 충분히 눌러 물기 제거, 단단한 두부 사용. (4) 소스가 너무 짜거나 맵다 → 식초/물 소량으로 풀고 
유자청/조청으로 밸런스. (5) 살사가 생강 향에 묻혀 텁텁함 → 생강을 과다 사용하지 말고 유자청·깻잎으로 상큼/초록 향을
보강.'
)

해당 출력은 Pydantic 클래스 형식으로 생성됩니다.   
with_structured_output 기능을 지원하지 않는 경우, PydanticOutputParser를 사용해야 합니다.

In [18]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object = Recipe)

recipe_template4 =ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
])

structured_llm2 = recipe_template4.partial(instruction = pydantic_parser.get_format_instructions()) | gpt_llm | pydantic_parser

response = structured_llm2.invoke('커피')
response

Recipe(name='에스프레소-미소 카라멜 된장글레이즈 연어 타코 (커피+일식+멕시칸 퓨전)', difficulty='중상 (불 조절과 글레이즈 농도 조절이 핵심)', origin='퓨전: 일본(미소/된장) + 멕시코(타코) + 이탈리아(에스프레소) + 북유럽(연어)', ingredients=['연어 필렛 300g (껍질 있는 것 추천)', '소금 3g', '후추 약간', '올리브오일 1큰술', '또띠아(옥수수 또는 밀) 6장', '양배추 채 1컵', '라임 1개', '고수(선택) 한 줌', '에스프레소 60ml (또는 진하게 내린 커피 80ml)', '된장(미소) 2큰술', '흑설탕 또는 황설탕 2큰술', '버터 1큰술', '간장 1큰술', '식초(사과식초/쌀식초) 1작은술', '고춧가루 또는 칠리플레이크 1/2작은술', '마요네즈 3큰술', '플레인 요거트 2큰술', '커피가루(곱게 분쇄) 1/2작은술 (소스용, 선택)', '마늘 1쪽(다짐) 또는 마늘가루 1/3작은술'], instructions=['연어에 소금·후추로 밑간하고 10분 두어 표면 수분을 정리한다(키친타월로 톡톡).', '글레이즈 만들기: 작은 팬에 에스프레소, 된장, 설탕, 간장, 식초, 마늘, 칠리를 넣고 중약불에서 저어가며 끓인다.', '거품이 잦아들고 점도가 시럽처럼 걸쭉해지면(숟가락 뒷면을 코팅하는 정도) 불을 끄고 버터를 넣어 윤기를 낸다. 너무 되면 물 1큰술로 조절한다.', '소스(크레마) 만들기: 마요네즈+요거트를 섞고, 라임즙 1큰술과 라임 제스트 약간을 넣는다. 선택으로 커피가루 1/2작은술을 아주 소량만 넣어 쌉쌀한 뒷맛을 만든다.', '팬을 강불로 예열 후 올리브오일을 두른다. 연어는 껍질면부터 3~4분 굽고, 뒤집어 1~2분만 익힌다(과익 방지).', '불을 중약불로 낮추고 글레이즈를 연어 위에 2~3번 얇게 덧발라가며 30~60초만 빠르게 코팅한다(타지 않게).', '연어를 꺼내 2분 휴지 후 먹기 좋은 크기로 찢거나 썬다.', '또띠아를 마른 팬에 10~20초

## [실습] LLM으로 보고서 개요 생성 후 섹션별 글 작성하기

Structured Output 구조를 활용해, LLM이 주제에 대한 구획을 먼저 구성하고    
해당 구획을 반복문이나 Batch로 각각 입력하여 긴 글을 쓰도록 만들어 보세요.

1. with_structured_output을 통해 주제에 대한 구획 작성하는 체인 `outliner` 만들기
2. 섹션별 글 작성 체인 `writer` 만들기
3. 반복문이나 batch()를 통해 `outliner`의 결과물을 `writer`에 전달하기
4. 최종 결과물 합치기

In [19]:
class Sections(BaseModel):
    topic: str = Field(description="글쓰기 주제")
    sections: list[str] = Field(description="주제에 대한 세부 섹션 개요 리스트 (최대 5개 섹션)")

In [20]:
outliner = gpt_llm.with_structured_output(Sections)
outline = outliner.invoke("""
멀티모달 LLM의 발전 과정에 대한 보고서 개요와 목차를 써줘.
각각의 개요는 병렬적 작성이 가능하도록 독립적인 내용을 담아야 하고.
개요만 보고도 내용이 구체적으로 드러나야 해.
목차는 리스트로 작성해.
""")
outline

Sections(topic='멀티모달 LLM의 발전 과정(연대기·기술축 중심) 보고서 개요 및 목차', sections=['정의·범위·평가 관점(멀티모달이 ‘무엇을’ 해결하는가): 텍스트 중심 LLM 대비 입력/출력 모달리티(이미지·오디오·비디오·센서·문서), 과업군(VQA, 캡셔닝, OCR/문서 이해, 시청각 대화, 비디오 내 이벤트 추적), 핵심 난제(정렬·추론·시간축·메모리·도구사용)와 대표 벤치마크(MMBench, SEED, MMMU, MathVista, VQA, TextVQA, DocVQA 등)로 ‘발전’을 측정하는 기준을 먼저 고정', '초기 멀티모달 신경망→비전-언어 사전학습(2018~2021): CNN/Transformer 기반 캡셔닝·VQA에서 시작해, 대규모 이미지-텍스트 사전학습(CLIP류의 대조학습, ALIGN 등)과 cross-attention 기반 비전-언어 모델(VilBERT/UNITER류)로 전환되며 ‘공통 임베딩 공간’과 ‘정렬 학습’이 성능·전이의 기반이 된 과정, 데이터(웹 스크랩 이미지-자막)와 학습 목표(contrastive, masked modeling)의 영향 비교', 'LLM 결합의 1세대(2022~2023): ‘비전 인코더+LLM’ 어댑터 패러다임—고정/미세조정된 비전 인코더 출력(특징 토큰)을 프로젝터(선형/MLP/Q-Former)로 LLM 입력 공간에 매핑해 시각 대화·지시따르기(Instruction Following)를 구현한 흐름(Flamingo, BLIP-2, LLaVA 계열), 정렬 데이터(이미지-설명→대화형 지시 데이터) 제작과 SFT/RLHF 도입이 사용자 경험을 어떻게 바꿨는지, 그리고 환각·세밀 인식 한계가 어디서 발생했는지', '고도화 단계(2023~2025): 고해상도·문서·비디오·오디오로의 확장과 ‘에이전트화’—(1) 이미지: 고해상도 타일링/멀티스케일, OCR 결합, 레이아웃 이해; (2) 비디오: 시간적 토큰 압축, 이벤트 세그먼트, 장기 컨텍스트; (3) 오디오: 음성인식/이해/생성의 통

In [21]:
outline.sections

['정의·범위·평가 관점(멀티모달이 ‘무엇을’ 해결하는가): 텍스트 중심 LLM 대비 입력/출력 모달리티(이미지·오디오·비디오·센서·문서), 과업군(VQA, 캡셔닝, OCR/문서 이해, 시청각 대화, 비디오 내 이벤트 추적), 핵심 난제(정렬·추론·시간축·메모리·도구사용)와 대표 벤치마크(MMBench, SEED, MMMU, MathVista, VQA, TextVQA, DocVQA 등)로 ‘발전’을 측정하는 기준을 먼저 고정',
 '초기 멀티모달 신경망→비전-언어 사전학습(2018~2021): CNN/Transformer 기반 캡셔닝·VQA에서 시작해, 대규모 이미지-텍스트 사전학습(CLIP류의 대조학습, ALIGN 등)과 cross-attention 기반 비전-언어 모델(VilBERT/UNITER류)로 전환되며 ‘공통 임베딩 공간’과 ‘정렬 학습’이 성능·전이의 기반이 된 과정, 데이터(웹 스크랩 이미지-자막)와 학습 목표(contrastive, masked modeling)의 영향 비교',
 'LLM 결합의 1세대(2022~2023): ‘비전 인코더+LLM’ 어댑터 패러다임—고정/미세조정된 비전 인코더 출력(특징 토큰)을 프로젝터(선형/MLP/Q-Former)로 LLM 입력 공간에 매핑해 시각 대화·지시따르기(Instruction Following)를 구현한 흐름(Flamingo, BLIP-2, LLaVA 계열), 정렬 데이터(이미지-설명→대화형 지시 데이터) 제작과 SFT/RLHF 도입이 사용자 경험을 어떻게 바꿨는지, 그리고 환각·세밀 인식 한계가 어디서 발생했는지',
 '고도화 단계(2023~2025): 고해상도·문서·비디오·오디오로의 확장과 ‘에이전트화’—(1) 이미지: 고해상도 타일링/멀티스케일, OCR 결합, 레이아웃 이해; (2) 비디오: 시간적 토큰 압축, 이벤트 세그먼트, 장기 컨텍스트; (3) 오디오: 음성인식/이해/생성의 통합; (4) 도구사용: 브라우징·코드·검색·OCR·ASR을 호출하는 멀티모달 에이전트, 그리고 모델 단독 vs 툴 

In [22]:
writer_prompt = ChatPromptTemplate([
    ('human','''보고서 주제에 대해, 하나의 섹션에 대한 전문적인 글을 작성하세요.

제목은 ##, 소목차는 ###으로 쓰고, 이외의 목차 형식은 넣지 마세요.
챕터명에 숫자를 넣지 마세요.
내용은 '입니다' 와 같은 말투로 작성하세요.

---
보고서 전체 주제: {topic}
세부 섹션 주제: {section}
''')
])
writer = writer_prompt | gpt_llm | StrOutputParser()
writer.invoke({'topic':outline.topic, 'section':outline.sections[0]})

'## 정의·범위·평가 관점(멀티모달이 ‘무엇을’ 해결하는가)\n\n멀티모달 LLM은 텍스트만을 입력·출력으로 다루는 기존 LLM의 범위를 넘어, 이미지·오디오·비디오·문서·센서 등 서로 다른 형식의 정보를 동일한 추론 프레임 안에서 이해하고, 이를 기반으로 언어적 응답 또는 다른 모달리티의 출력을 생성하는 모델 계열입니다. 본 보고서의 연대기·기술축 중심 서술을 위해서는 “무엇이 멀티모달의 발전인가”를 먼저 고정하는 작업이 필수입니다. 즉, 입력·출력 모달리티의 확장, 해결 가능한 과업군의 확대, 그리고 정렬·추론·시간축 처리와 같은 핵심 난제의 완화 정도를 벤치마크로 계량화하는 평가 관점이 선행 정의되어야 합니다.\n\n### 멀티모달의 정의와 범위\n\n멀티모달이 해결하는 핵심 문제는 “비언어 정보의 구조화”와 “언어 추론과의 결합”입니다. 텍스트 중심 LLM은 언어로 기술된 세계를 강하게 모델링하지만, 실제 환경의 많은 정보는 픽셀·파형·프레임 시퀀스·레이아웃·센서 시계열로 존재합니다. 멀티모달 LLM은 이러한 비정형 신호를 언어 추론이 가능한 중간 표현으로 정렬하고, 질문응답·설명·계획·행동으로 연결하는 통합 인터페이스를 제공하는 것이 범위입니다.\n\n또한 멀티모달은 단순히 “이미지를 입력받는 LLM”에 한정되지 않습니다. 문서(레이아웃과 텍스트가 공존), 차트·도표, 음성(대화·감정·화자 정보), 비디오(시간축 사건과 인과), 로보틱스 센서(힘·촉각·IMU·위치)까지 포함하는 것이 합리적 범위입니다. 다만 본 보고서는 멀티모달 LLM의 발전 과정을 기술축으로 비교하기 위해, 입력 모달리티 확장과 추론 능력의 결합을 중심으로 범위를 설정하고, 생성형 비디오·오디오 자체의 품질 경쟁은 부차적 축으로 다루는 것이 타당합니다.\n\n### 텍스트 중심 LLM 대비 입력·출력 모달리티 확장\n\n멀티모달 LLM의 확장은 입력과 출력 양쪽에서 관측됩니다.\n\n- 이미지 입력은 시각 인식(객체·속성·관계), 공간 추론, 텍스트 포함 이미지의 판독(OCR 연계

In [23]:
writer = writer_prompt.partial(topic=outline.topic) | gpt_llm | StrOutputParser()
# topic을 미리 채워 매개변수 1개
result = writer.batch(outline.sections)
result

['## 정의·범위·평가 관점(멀티모달이 ‘무엇을’ 해결하는가)\n\n### 멀티모달 LLM의 정의와 문제 설정\n멀티모달 LLM은 텍스트 기반 대규모 언어모델을 중심으로, 이미지·오디오·비디오·문서·센서 등 서로 다른 형태의 입력을 함께 받아들이고, 텍스트를 포함한 다양한 형태의 출력까지 생성할 수 있도록 확장한 모델 계열입니다. 핵심은 “언어로만 기술되지 않은 정보”를 모델의 인지·추론·의사결정 과정에 포함시키는 데 있으며, 이는 현실 세계의 관찰(시각·청각·계측)과 지식(언어·문서)을 통합해 과업을 해결하는 문제로 정식화됩니다. 따라서 멀티모달은 단순히 입력 채널을 늘리는 기능 확장이 아니라, 서로 다른 모달리티가 제공하는 상보적 단서들을 정렬하고, 이를 기반으로 추론하며, 필요 시 도구를 호출해 외부 정보를 결합하는 종합 능력의 확장을 의미합니다.\n\n### 텍스트 중심 LLM 대비 입력·출력 모달리티 확장 범위\n텍스트 중심 LLM이 주로 텍스트 입력→텍스트 출력의 범위에서 언어적 패턴 학습과 지식 추론을 수행한다면, 멀티모달 LLM은 입력의 관측 공간과 출력의 표현 공간을 동시에 확장합니다. 입력 측면에서 이미지는 공간적 구조·객체·시각적 관계를 제공하며, 오디오는 발화 내용뿐 아니라 화자 특성·억양·환경음을 포함한 시간적 신호를 제공합니다. 비디오는 시간축을 따라 변화하는 장면·행동·상호작용을 포함하므로, 단일 이미지보다 높은 수준의 이벤트 이해가 필요합니다. 문서(스캔·PDF·표·도식)는 텍스트와 레이아웃, 타이포그래피, 표 구조가 결합된 형태이므로 단순 OCR을 넘어 문서 이해가 요구됩니다. 센서(예: IMU, LiDAR, 생체신호, 산업 계측)는 연속적 수치 신호 또는 3차원 구조를 포함하며, 노이즈·결측·정규화 이슈가 동반됩니다. 출력 또한 텍스트 응답을 기본으로 하되, 좌표 기반 지시(바운딩 박스), 구조화된 표/JSON, 오디오 합성, 이미지 생성/편집, 비디오 요약 등으로 확장될 수 있으며, 평가 시에는 이러한 출력 형식의 정확성과 

In [24]:
draft = '\n\n'.join(result)
with open('result.md', 'w', encoding='utf-8') as f:
    f.write(draft)

print(draft[0:100])

## 정의·범위·평가 관점(멀티모달이 ‘무엇을’ 해결하는가)

### 멀티모달 LLM의 정의와 문제 설정
멀티모달 LLM은 텍스트 기반 대규모 언어모델을 중심으로, 이미지·오디오·


<br><br>
## Runnables

LangChain 체인의 기본 구조는 `RunnableSequence` 클래스로 구성됩니다.   

이 때, 시퀀스를 구성한 llm, prompt, chain 각 모듈은 Runnables에 해당합니다.   
Runnables은 자유롭게 체인에 포함되어 결과를 연결할 수 있습니다.



이번에는, 데이터 흐름을 제어하는 특별한 Runnable인   
RunnablePassthrough와 RunnableParallel을 이용해 체인을 구성해 보겠습니다.


<br><br>
### RunnablePassthrough
RunnablePassthrough는 체인의 직전 출력을 그대로 가져옵니다.

In [25]:
from langchain_core.runnables import RunnablePassthrough

prompt1 = ChatPromptTemplate(["{director}의 대표 작품은 무엇입니까? 하나의 작품만 선택하고, 해당 작품에 대해 20자 이내로 설명하세요."])
chain1 = (
    prompt1
    | gpt_llm
    | StrOutputParser()
    | {'answer': RunnablePassthrough()})

response = chain1.invoke("봉준호")
response

{'answer': '**기생충**: 계층 갈등을 그린 블랙코미디'}

<br><br>
### RunnableParallel

RunnableParallel은 서로 다른 체인을 병렬적으로 실행하여 dict 구조로 전달합니다.

In [26]:
from langchain_core.runnables import RunnableParallel

prompt1 = ChatPromptTemplate(["색깔을 하나 알려주세요, 색깔만 출력하세요."])
prompt2 = ChatPromptTemplate(["음식을 하나 알려주세요, 음식만 출력하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(color = chain1, food = chain2)
# 개별 체인을 병렬 실행한 뒤, dict로 반환

chain3.invoke({})

{'color': '파란색', 'food': '비빔밥'}

## Assign()

RunnableParallel을 사용하면 중간 체인의 결과를 전달하여, 다음 체인의 결과를 함께 얻을 수 있습니다.   

In [27]:
prompt1 = ChatPromptTemplate(["잭슨빌은 어느 나라의 도시입니까? 나라 이름만 출력"])
prompt2 = ChatPromptTemplate(
    ["{country}의 대표적인 인물 3명을 나열하세요. 인물의 이름만 출력하세요."]
)

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(country = chain1).assign(people = chain2)
#        {    'country': '미국'           }  +    {'people': chain2(미국)}

chain3.invoke({})

{'country': '미국', 'people': '조지 워싱턴  \n에이브러햄 링컨  \n마틴 루터 킹 주니어'}

<br><br><br><br><br><br><br><br>
chain2에서 새로운 매개변수가 추가되는 경우는 어떻게 해야 할까요?

In [28]:
prompt1 = ChatPromptTemplate([
    "{city}는 어느 나라의 도시인가요? 나라 이름만 출력하세요."])
prompt2 = ChatPromptTemplate([
    "{country}의 유명한 인물은 누가 있나요? {num} 명의 이름을 나열하세요. 사람 이름만 ,로 구분하여 나열하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()

chain2 = (
    RunnablePassthrough.assign(country = chain1)
    # 입력받은 city, num에 country를 추가하여 전달
    # city, num          +    country

    | prompt2
    # country, num을 받아 실행
    | gpt_llm
    | StrOutputParser()
)

print(chain2.invoke({"city": "잭슨빌", "num": "3"}))

조지 워싱턴, 에이브러햄 링컨, 마틴 루터 킹 주니어


<br><br>
assign을 여러 개 연결할 수 있습니다.

In [29]:
chain4 = (prompt2
    | gpt_llm
    | StrOutputParser())

chain3 = RunnablePassthrough.assign(country = chain1).assign(people = chain4)
#        {'city', 'num'}      +    {'country'}         +    {'people'}

chain3.invoke({"city": "부에노스 아이레스", "num": "3"})

{'city': '부에노스 아이레스',
 'num': '3',
 'country': '아르헨티나',
 'people': '리오넬 메시, 디에고 마라도나, 에바 페론'}

<br><br><br>JsonOutputParser를 쓴다면 아래와 같이 만들 수도 있습니다.

In [30]:
prompt1 = ChatPromptTemplate(
    ["영화 배우 한 명과 대표작 하나를 출력하세요. json 형식으로 출력하고, 각 항목은 actor, movie로 표시하세요."])
prompt2 = ChatPromptTemplate(["{actor}는 {movie}에서 어떤 역할을 했습니까?"])

chain1 = prompt1 | gpt_llm | JsonOutputParser()
# {actor, movie}
chain2 =(
     chain1 | prompt2 | gpt_llm | StrOutputParser()
)
chain2.invoke({})

'송강호는 영화 **〈기생충〉(2019)**에서 **김기택** 역을 맡았습니다.  \n김기택은 기우(최우식)·기정(박소담)의 아버지이자 김가족의 가장으로, 박사장(이선균) 집에 **운전기사로 취업**하며 이야기를 이끌어가는 핵심 인물입니다.'

In [31]:
chain3 = prompt2 | gpt_llm | StrOutputParser()

chain4 = chain1.assign(result = chain3)

chain4.invoke({})

{'actor': '송강호',
 'movie': '기생충',
 'result': '송강호는 영화 **〈기생충〉(2019)**에서 **기택(김기택)** 역을 맡았습니다.  \n가난한 가족의 **아버지**로, 박사장(이선균) 집에 **운전기사로 취업**하면서 가족들이 차례로 그 집에 들어가게 되는 과정의 중심에 있는 인물입니다.'}